# Huấn luyện model nhận diện ROCK / PAPER / SCISSORS

Chọn kernel Python 3.12, chạy các cell theo thứ tự. Cell đầu tự cài thư viện đúng kernel; cell dataset tự tải train/test chính thức (khoảng 220 MB), không cần Kaggle hay script bên ngoài. Dùng Keras 3 backend PyTorch, cùng backend notebook game.

Notebook này **huấn luyện**, không cần tải model pretrained trước. Game đã có checkpoint `models/rps.h5`, không cần chạy training để chơi. Kết quả train mới lưu riêng vào `models/rps-trained.h5`; muốn thử trong game, chỉnh `rps_model_path` ở Cell 2. Train 25 epoch có thể mất nhiều phút hoặc lâu hơn trên CPU.

Giữ đúng RGB 150×150, nearest, chia 255 và nhãn alphabet paper/rock/scissors. Nguồn dataset: https://www.tensorflow.org/datasets/catalog/rock_paper_scissors


In [ ]:
# Thiết lập máy mới: dùng cùng Keras/PyTorch như notebook game.
from pathlib import Path
import os
import sys
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "PythonExample/AmazingHand_Setup.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục repo AmazingHand.")
os.environ["KERAS_BACKEND"] = "torch"
os.environ["USE_TF"] = "0"
sys.path.insert(0, str(PROJECT_ROOT / "PythonExample"))
from AmazingHand_Setup import ensure_dependencies, ensure_training_dataset
ensure_dependencies(PROJECT_ROOT / "requirements-training.txt")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
%matplotlib inline
import torch
torch.set_num_threads(min(8, os.cpu_count() or 1))
import keras
if keras.backend.backend() != "torch":
    raise RuntimeError("Restart Kernel và chạy lại từ cell đầu để chọn backend torch.")
from keras.src.legacy.preprocessing.image import ImageDataGenerator
from keras.utils import img_to_array, load_img
import random
print("PASS — Thư viện training sẵn sàng; kernel:", sys.executable)


## Explore the Data

In [ ]:
# Tự tải bộ train/test chính thức TensorFlow nếu chưa có (khoảng 220 MB).
DATASET_DIR = ensure_training_dataset(PROJECT_ROOT)
print("Dataset:", DATASET_DIR)

TRAINING_DIR = str(DATASET_DIR / 'train')
VALIDATION_DIR = str(DATASET_DIR / 'test')

rock_dir = os.path.join(TRAINING_DIR, 'rock')
paper_dir = os.path.join(TRAINING_DIR, 'paper')
scissors_dir = os.path.join(TRAINING_DIR, 'scissors')

print('Total training rock images:', len(os.listdir(rock_dir)))
print('Total training paper images:', len(os.listdir(paper_dir)))
print('Total training scissors images:', len(os.listdir(scissors_dir)))

rock_files = os.listdir(rock_dir)
print(rock_files[:10])

paper_files = os.listdir(paper_dir)
print(paper_files[:10])

scissors_files = os.listdir(scissors_dir)
print(scissors_files[:10])

## Data Preprocessing

In [ ]:
training_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest')

validation_datagen = ImageDataGenerator(rescale = 1./255)

train_generator = training_datagen.flow_from_directory(
    TRAINING_DIR,
    target_size=(150,150),
    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=(150,150),
    class_mode='categorical')
print("Nhãn:", train_generator.class_indices)
assert train_generator.class_indices == {"paper": 0, "rock": 1, "scissors": 2}

## Building Model

In [ ]:
model = keras.models.Sequential([
    # Note the input shape is the desired size of the image 150x150 with 3 bytes color
    # This is the first convolution
    keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(150, 150, 3)),
    keras.layers.MaxPooling2D(2, 2),
    # The second convolution
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    # The third convolution
    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    # The fourth convolution
    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),
    # Flatten the results to feed into a DNN
    keras.layers.Flatten(),
    keras.layers.Dropout(0.5),
    # 512 neuron hidden layer
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(3, activation='softmax')
])

In [ ]:
model.summary()

In [ ]:
model.compile(loss = 'categorical_crossentropy', optimizer='rmsprop', metrics=['accuracy'])

## Training

In [ ]:
history = model.fit(
    train_generator,
    epochs=25,
    validation_data=validation_generator,
    verbose=1,
)

# Lưu bản train mới riêng; không ghi đè checkpoint đang chơi game.
TRAINED_MODEL_PATH = PROJECT_ROOT / "models/rps-trained.h5"
TRAINED_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
model.save(TRAINED_MODEL_PATH)
print("Đã lưu:", TRAINED_MODEL_PATH)
print("Thứ tự nhãn:", train_generator.class_indices)

## Evaluating Accuracy and Loss for the Model

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(loss) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'r', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'r', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()
